In [ ]:
#| default_exp renderers.flux_klein

# renderers.flux_klein

> Renderer for Flux Klein 9B via Replicate.
>
> Supports LoRA and negative prompts. No reference image input.
> API key: `REPLICATE_API_TOKEN` environment variable.
> Set `renderer.flux_klein.model_ref` in your config to the Replicate model string.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import os
from pathlib import Path

from manhualizer.config import OutputConfig, RendererConfig
from manhualizer.models import Panel, RenderResult
from manhualizer.render import BaseRenderer, ModelSpec

In [ ]:
#| export
class FluxKleinRenderer(BaseRenderer):
    """Image generation via Flux Klein 9B on Replicate.

    Supports LoRA weights and negative prompts.
    No reference image input — character consistency via text prompts only.

    Requires: REPLICATE_API_TOKEN environment variable.
    Config: renderer.flux_klein.model_ref (Replicate model string)
    """

    def __init__(self, model_spec: ModelSpec, config: RendererConfig):
        super().__init__(model_spec, config)
        self._model_cfg = config.flux_klein

    def _client(self):
        import replicate  # type: ignore
        if not os.environ.get("REPLICATE_API_TOKEN"):
            raise EnvironmentError("REPLICATE_API_TOKEN is not set")
        return replicate

    def _build_input(self, panel: Panel, output_cfg: OutputConfig, negative_prompt: str = "") -> dict:
        inp: dict = {
            "prompt": panel.visual_prompt,
            "num_outputs": 1,
            "output_format": output_cfg.format,
        }
        # Prefer aspect_ratio string; fall back to width/height
        if output_cfg.aspect_ratio:
            inp["aspect_ratio"] = output_cfg.aspect_ratio
        else:
            w, h = output_cfg.resolved_dimensions()
            inp["width"] = w
            inp["height"] = h
        if negative_prompt:
            inp["negative_prompt"] = negative_prompt
        if self.config.loras:
            inp["extra_loras"] = [
                {"url": lora.path, "scale": lora.strength}
                for lora in self.config.loras
            ]
            # Prepend trigger words to the prompt
            trigger_words = " ".join(
                lora.trigger_word for lora in self.config.loras if lora.trigger_word
            )
            if trigger_words:
                inp["prompt"] = f"{trigger_words}, {inp['prompt']}"
        return inp

    async def render_async(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None = None,
        negative_prompt: str = "",
    ) -> RenderResult:
        import asyncio
        return await asyncio.get_event_loop().run_in_executor(
            None, self._render_sync, panel, output_dir, output_cfg, negative_prompt
        )

    def _render_sync(
        self, panel: Panel, output_dir: Path, output_cfg: OutputConfig, negative_prompt: str = ""
    ) -> RenderResult:
        import httpx
        replicate = self._client()

        if not self._model_cfg.model_ref:
            raise ValueError(
                "renderer.flux_klein.model_ref is not set. "
                "Set it to your Replicate model string (e.g. 'owner/model:version')."
            )

        inp = self._build_input(panel, output_cfg, negative_prompt)
        output = replicate.run(self._model_cfg.model_ref, input=inp)

        url = output[0] if isinstance(output[0], str) else output[0].url
        image_bytes = httpx.get(url).content

        out_path = output_dir / f"panel_{panel.panel_number:04d}.{output_cfg.format}"
        out_path.write_bytes(image_bytes)

        return RenderResult(
            panel_number=panel.panel_number,
            image_path=out_path,
            backend_used=self.model_spec.name,
            prompt_used=inp["prompt"],
            metadata={"model_ref": self._model_cfg.model_ref, "loras": len(self.config.loras)},
        )

In [ ]:
# Construction and input-building tests (no API call)
from manhualizer.render import MODELS
from manhualizer.config import RendererConfig, LoRAConfig, OutputConfig
from manhualizer.models import Panel
from manhualizer.renderers.flux_klein import FluxKleinRenderer

cfg = RendererConfig(loras=[LoRAConfig(path="https://example.com/style.safetensors",
                                       strength=0.8, trigger_word="mystyle")])
renderer = FluxKleinRenderer(MODELS["flux-klein"], cfg)
assert renderer.model_spec.capabilities.lora
assert not renderer.model_spec.capabilities.reference_images

panel = Panel(panel_number=1, scene_id="s1", location="Forest",
              action_description="walks", visual_prompt="manhua style, forest")

# With aspect_ratio set → passes aspect_ratio string, no width/height
out_cfg = OutputConfig(aspect_ratio="9:16")
inp = renderer._build_input(panel, out_cfg, negative_prompt="blurry")
assert inp["aspect_ratio"] == "9:16"
assert "width" not in inp
assert "height" not in inp
assert inp["negative_prompt"] == "blurry"
assert len(inp["extra_loras"]) == 1
assert inp["extra_loras"][0]["url"] == "https://example.com/style.safetensors"
assert inp["prompt"].startswith("mystyle,")  # trigger word prepended

# Without aspect_ratio → falls back to width/height
out_cfg2 = OutputConfig(aspect_ratio="", width=512, height=768)
inp2 = renderer._build_input(panel, out_cfg2)
assert inp2["width"] == 512
assert inp2["height"] == 768
assert "aspect_ratio" not in inp2

print("FluxKleinRenderer OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()